In [16]:
!pip install openai gtts python-dotenv

In [20]:
import os

os.environ["OPENAI_API_KEY"] = "Insira sua chave aqui"
print("Chave definida com sucesso!")

Chave definida com sucesso!


In [23]:
import os
import base64
from openai import OpenAI
from gtts import gTTS
from IPython.display import Javascript, display, Audio
from google.colab import output

client = OpenAI()

AUDIO_INPUT = "audio_input.wav"
AUDIO_OUTPUT = "audio_output.mp3"

# ==============================
# FUNÇÃO PARA GRAVAR ÁUDIO
# ==============================
def gravar_audio(duracao=5):
    display(Javascript(f"""
    async function recordAudio() {{
      const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();
      await new Promise(resolve => setTimeout(resolve, {duracao*1000}));
      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);
      const blob = new Blob(chunks);
      const arrayBuffer = await blob.arrayBuffer();
      const base64Audio = btoa(
        new Uint8Array(arrayBuffer)
          .reduce((data, byte) => data + String.fromCharCode(byte), '')
      );
      return base64Audio;
    }}
    recordAudio();
    """))

    audio_base64 = output.eval_js("recordAudio()")
    audio_bytes = base64.b64decode(audio_base64)

    with open(AUDIO_INPUT, "wb") as f:
        f.write(audio_bytes)

# ==============================
# LOOP DO ASSISTENTE
# ==============================

historico = [
    {"role": "system", "content": "Você é um assistente virtual inteligente e objetivo."}
]

print("🎙️ Assistente iniciado! Diga 'sair' para encerrar.\n")

while True:

    print("🎤 Gravando pergunta...")
    gravar_audio(5)

    # Transcrição
    with open(AUDIO_INPUT, "rb") as audio:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio
        )

    pergunta = transcript.text
    print("🗣️ Você disse:", pergunta)

    if "sair" in pergunta.lower():
        print("👋 Encerrando assistente.")
        break

    historico.append({"role": "user", "content": pergunta})

    # GPT responde
    resposta = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=historico
    )

    texto_resposta = resposta.choices[0].message.content
    print("🤖 Assistente:", texto_resposta)

    historico.append({"role": "assistant", "content": texto_resposta})

    # Converter resposta em voz
    tts = gTTS(text=texto_resposta, lang="pt")
    tts.save(AUDIO_OUTPUT)

    display(Audio(AUDIO_OUTPUT))

🎙️ Assistente iniciado! Diga 'sair' para encerrar.

🎤 Gravando pergunta...


<IPython.core.display.Javascript object>

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}